# eda all roads

slow = speed <= 60% of free speed (our cutoff, not theirs). next link = next order_index (only checked on D12_I5_N)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REL = Path("/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public")
ROADS = sorted(p.name for p in (REL / "corridors").iterdir() if p.is_dir())
SLOW_RATIO = 0.60
print(len(ROADS), "roads:", ROADS)


def topology(road):
    t = pd.read_csv(REL / "corridors" / road / "network" / "lwr_mainline_topology.csv")
    t["link_id"] = t["link_id"].astype(str)
    return t.drop_duplicates("link_id").sort_values("order_index").reset_index(drop=True)


def train_files(road, masked=False):
    sub = "mainline_states_masked" if masked else "mainline_states"
    return sorted((REL / "corridors" / road / "train" / sub).rglob("*.parquet"))

all train days, slow

In [ ]:
day_rows, hour_rows, head_rows, worst = [], [], [], {}

for road in ROADS:
    topo = topology(road)
    order = topo["link_id"].tolist()
    thr = (SLOW_RATIO * topo["free_speed_kmh"]).to_numpy()[:, None]
    free = topo.set_index("link_id")["free_speed_kmh"]
    heads = np.zeros(len(order))
    for f in train_files(road):
        d = pd.read_parquet(f, columns=["timestamp", "link_id", "speed_kmh"])
        d["link_id"] = d["link_id"].astype(str)
        d["timestamp"] = pd.to_datetime(d["timestamp"], utc=True)
        d = d[d["speed_kmh"].notna()]
        link_free = d["link_id"].map(free)
        d["slow"] = d["speed_kmh"] <= SLOW_RATIO * link_free
        date = d["timestamp"].iloc[0].normalize()
        share = d["slow"].mean()
        day_rows.append({"road": road, "date": date, "slow_share": share,
                         "max_speed": d["speed_kmh"].max(),
                         "above_free_share": (d["speed_kmh"] > link_free).mean()})
        h = d.groupby(d["timestamp"].dt.hour)["slow"].agg(["sum", "size"])
        h["road"] = road
        hour_rows.append(h.rename_axis("hour").reset_index())

        grid = d.groupby(["link_id", "timestamp"])["speed_kmh"].mean().unstack("timestamp").reindex(order).values
        heads[:-1] += np.nansum((grid[:-1] <= thr[:-1]) & (grid[1:] > thr[1:]), axis=1)
        if road not in worst or share > worst[road][0]:
            worst[road] = (share, date, grid)
    head_rows.append(pd.DataFrame({"road": road, "link_id": order,
                                   "order_index": topo["order_index"], "head_events": heads}))
    print("done", road)

days = pd.DataFrame(day_rows)
days["weekday"] = days["date"].dt.day_name().str[:3]
days["weekend"] = days["date"].dt.dayofweek >= 5
hours = pd.concat(hour_rows, ignore_index=True).groupby(["road", "hour"])[["sum", "size"]].sum()
heads = pd.concat(head_rows, ignore_index=True)

per road

In [ ]:
def top5(g):
    g = g.sort_values("head_events", ascending=False)
    return pd.Series({"top5_head_share": g["head_events"].head(5).sum() / max(g["head_events"].sum(), 1),
                      "top5_head_links": ", ".join(g["link_id"].head(5))})


summary = days.groupby("road").agg(
    days=("date", "size"),
    slow_mean=("slow_share", "mean"),
    slow_median=("slow_share", "median"),
    slow_max=("slow_share", "max"),
    zero_jam_days=("slow_share", lambda s: int((s == 0).sum())),
    max_speed=("max_speed", "max"),
    above_free_share=("above_free_share", "mean"),
)
wk = days.groupby(["road", "weekend"])["slow_share"].mean().unstack()
summary["weekday_slow"] = wk[False]
summary["weekend_slow"] = wk[True]
summary = summary.join(heads.groupby("road")[["link_id", "head_events"]].apply(top5))
pd.set_option("display.max_colwidth", 80)
display(summary.round(4))
summary.to_csv("/kaggle/working/eda_road_summary.csv")

weekday / hour / day to day

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
by_wd = days.pivot_table(index="road", columns="weekday", values="slow_share", aggfunc="mean")
by_wd = by_wd.reindex(columns=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]) * 100
im = axes[0].imshow(by_wd.values, aspect="auto", cmap="YlOrRd")
axes[0].set(xticks=range(7), xticklabels=by_wd.columns, yticks=range(len(by_wd)), yticklabels=by_wd.index,
            title="Slow readings by weekday (%)")
fig.colorbar(im, ax=axes[0])
by_h = (hours["sum"] / hours["size"] * 100).unstack("hour").reindex(columns=range(24))
im = axes[1].imshow(by_h.values, aspect="auto", cmap="YlOrRd")
axes[1].set(xticks=range(24), yticks=range(len(by_h)), yticklabels=by_h.index,
            xlabel="hour (UTC)", title="Slow readings by hour (%)")
fig.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(14, 5))
for road, g in days.sort_values("date").groupby("road"):
    ax.plot(g["date"], g["slow_share"] * 100, lw=0.8, label=road)
ax.set(ylabel="slow readings (%)", title="Day-to-day spread per road")
ax.legend(ncol=5, fontsize=8)
plt.show()

worst day each road

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(24, 9))
for ax, road in zip(axes.flat, ROADS):
    share, date, grid = worst[road]
    ax.imshow(grid, aspect="auto", cmap="RdYlGn", vmin=20, vmax=120)
    ax.set_title(f"{road}  {date.date()}  slow {share:.1%}", fontsize=9)
    ax.set_xlabel("5-min slot")
axes[0, 0].set_ylabel("link (order_index)")
axes[1, 0].set_ylabel("link (order_index)")
plt.suptitle("Worst train day per road: speed along the road over the day")
plt.tight_layout()
plt.show()

hidden cells jams more often?

In [ ]:
rows = []
cols = ["timestamp", "station_id", "link_id", "speed_kmh"]
for road in ROADS:
    free = topology(road).set_index("link_id")["free_speed_kmh"]
    originals = {f.name: f for f in train_files(road)}
    masked = train_files(road, masked=True)
    for regime in ["R1", "R2", "R3"]:
        for mf in [f for f in masked if f.parent.name == f"mask_regime={regime}"][-5:]:
            o = pd.read_parquet(originals[mf.name], columns=cols)
            m = pd.read_parquet(mf, columns=cols)
            x = o.merge(m, on=cols[:3], suffixes=("_true", "_masked"))
            x = x[x["speed_kmh_true"].notna()]
            slow = x["speed_kmh_true"] <= SLOW_RATIO * x["link_id"].astype(str).map(free)
            hidden = x["speed_kmh_masked"].isna()
            rows.append({"road": road, "regime": regime, "hidden_if_slow": hidden[slow].mean(),
                         "hidden_if_fast": hidden[~slow].mean(), "slow_cells": int(slow.sum())})
mask = pd.DataFrame(rows).groupby(["road", "regime"]).mean().round(3)
mask["gap"] = (mask["hidden_if_slow"] - mask["hidden_if_fast"]).round(3)
display(mask)

has_sensor vs readings

In [ ]:
rows = []
for road in ROADS:
    topo = topology(road)
    d = pd.read_parquet(train_files(road)[-1], columns=["link_id", "speed_kmh"])
    n = d.assign(link_id=d["link_id"].astype(str)).groupby("link_id")["speed_kmh"].count()
    topo["readings"] = topo["link_id"].map(n).fillna(0)
    rows.append(topo.groupby("has_sensor")["readings"].agg(links="size", mean_readings="mean",
                                                           links_without_readings=lambda r: int((r == 0).sum()))
                    .assign(road=road).reset_index())
display(pd.concat(rows).set_index(["road", "has_sensor"]).round(1))

task 2 inputs

In [ ]:
rows = []
for road in sorted(p.name for p in (REL / "task2").iterdir() if p.is_dir()):
    for split in ["validation", "private"]:
        base = REL / "task2" / road / split
        if not base.is_dir():
            continue
        sheet = pd.read_csv(base / "sample_submission_queue.csv", usecols=["window_id"])
        hist = pd.read_parquet(base / "window_history.parquet", columns=["window_id", "pct_observed"])
        rows.append({"road": road, "split": split, "windows": sheet["window_id"].nunique(),
                     "queue_rows": len(sheet), "history_rows": len(hist),
                     "history_blank_share": round((hist["pct_observed"] == 0).mean(), 3)})
display(pd.DataFrame(rows))
print(pd.read_csv(base / "window_index.csv", nrows=5).to_string())